# Advanced RAG · Retrieval Levers (runnable on Bedrock)

Direct Amazon Bedrock, no LiteLLM, no framework. Every technique is a small, readable function you can run and inspect.

**What is inside**

| Lever | Techniques in this notebook |
|---|---|
| INDEX | semantic chunking, small-to-big, contextual retrieval |
| QUERY | multi-query / RAG-Fusion, HyDE, step-back |
| RANK | hybrid + Reciprocal Rank Fusion, LLM reranking, MMR, contextual compression |
| Compose | one "strong retriever" that stacks the best of them |

**Run it (VS Code)**
1. `python -m venv .venv` then activate it, and select `.venv` as the notebook kernel.
2. `pip install -U boto3 numpy rank-bm25`
3. Give it AWS creds: `aws configure`, or set `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` / `AWS_REGION`.
4. Run cells top to bottom.

**Run it (Google Colab)**
1. `!pip install -q boto3 numpy rank-bm25`
2. Set creds via Colab secrets or env vars: `os.environ["AWS_ACCESS_KEY_ID"] = ...`, and so on.
3. Run cells top to bottom.

Model: `us.anthropic.claude-haiku-4-5-20251001-v1:0` (the `us.` cross-region inference-profile prefix is required for on-demand Claude). Embeddings: `amazon.titan-embed-text-v2:0`.

In [ ]:
# If running fresh, uncomment the next line:
# %pip install -U boto3 numpy rank-bm25
import boto3, json, re
import numpy as np

In [ ]:
REGION = "us-east-1"
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. inference profile required
EMBED_MODEL = "amazon.titan-embed-text-v2:0"

brt = boto3.client("bedrock-runtime", region_name=REGION)

try:
    ident = boto3.client("sts", region_name=REGION).get_caller_identity()
    print("AWS creds OK. Account:", ident["Account"])
except Exception as e:
    print("No AWS creds detected. Pure-Python cells (BM25, RRF, MMR) still run.")
    print("Bedrock cells will work once creds are set. Detail:", str(e)[:160])

In [ ]:
# --- shared helpers: one chat call, one embed call, plus vector math ---

def chat(prompt, system=None, max_tokens=500, temperature=0.0):
    kwargs = dict(
        modelId=MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
    )
    if system:
        kwargs["system"] = [{"text": system}]
    resp = brt.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"].strip()

def embed(text):
    resp = brt.invoke_model(modelId=EMBED_MODEL, body=json.dumps({"inputText": text}))
    return np.array(json.loads(resp["body"].read())["embedding"], dtype=float)

def embed_many(texts):
    return np.vstack([embed(t) for t in texts])

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def cosine_scores(matrix, q):
    # matrix: (n, d), q: (d,) -> (n,) cosine of each row against q
    mn = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
    qn = q / (np.linalg.norm(q) + 1e-9)
    return mn @ qn

def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

# Why converse() and not raw invoke_model for chat: the Converse API gives one
# clean request/response shape for Claude and avoids hand-building the body JSON.
# Why we send only temperature (no top_p): Bedrock Claude rejects both together.

## The corpus

A small TravelMind knowledge base. Each entry is one document. We also build one deliberately mixed-topic paragraph (`HANDBOOK`) so semantic chunking has real topic shifts to find.

In [ ]:
CORPUS = {
  "refunds":    "TravelMind refunds. A cancelled flight is fully refundable to the original payment method. Refunds are processed within 7 to 10 business days. A voluntary cancellation by the passenger follows the fare rules of the ticket.",
  "change_fee": "Change fees by tier. Basic and Silver members pay a change fee to modify a booking. Gold and Platinum members have the change fee waived on eligible fares. A fare difference may still apply.",
  "tiers":      "Loyalty tiers. TravelMind has four tiers: Basic, Silver, Gold, and Platinum. Tier is based on miles flown in a calendar year. Gold unlocks waived change fees, priority boarding, and extra baggage.",
  "pnr":        "Booking reference. A PNR is a six character alphanumeric code that identifies a booking, for example JX48Q2. Use the PNR and the last name to retrieve or modify a reservation.",
  "checkin":    "Check in. Online check in opens 48 hours before departure and closes 60 minutes before for domestic flights. Airport check in counters close 45 minutes before departure.",
  "baggage":    "Baggage. Basic fares include one cabin bag up to 7 kg. Checked baggage allowance depends on fare and tier. Gold and above receive one extra checked bag.",
  "seat":       "Seat changes. Seats can be changed free of charge until check in, subject to availability. Preferred and extra legroom seats may carry a fee for Basic and Silver members.",
}

HANDBOOK = (
  "Boarding begins 40 minutes before departure and gates close 15 minutes before. "
  "Passengers should be at the gate on time with a boarding pass and ID. "
  "The loyalty program has four tiers based on annual miles flown. "
  "Gold and Platinum members enjoy waived change fees and priority boarding. "
  "Complimentary meals are served on flights longer than three hours. "
  "Special meals must be requested at least 24 hours before departure."
)

CHUNK_IDS = list(CORPUS.keys())
CHUNK_TEXTS = [CORPUS[k] for k in CHUNK_IDS]
print(len(CHUNK_IDS), "documents loaded")

## Baseline: fixed-size chunking

The naive default. It splits by word count and happily cuts mid-thought. Notice a boundary can land in the middle of a topic.

In [ ]:
def fixed_chunks(text, size=22, overlap=5):
    w = text.split()
    out, i = [], 0
    while i < len(w):
        out.append(" ".join(w[i:i+size]))
        if i + size >= len(w):
            break
        i += size - overlap
    return out

for c in fixed_chunks(HANDBOOK):
    print("-", c)

## INDEX lever 1 · Semantic chunking

Walk the sentences, embed each, and cut where consecutive similarity drops below a threshold. The boundary lands at a real topic change (boarding, then loyalty, then meals), not at an arbitrary word count.

Cost: one embedding per sentence at index time.

In [ ]:
def semantic_chunks(text, threshold=0.5, max_sentences=5):
    sents = sentences(text)
    if len(sents) <= 1:
        return sents
    vecs = embed_many(sents)                 # one vector per sentence
    chunks, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        drop = cosine(vecs[i-1], vecs[i]) < threshold   # meaning shifted
        if drop or len(cur) >= max_sentences:
            chunks.append(" ".join(cur)); cur = [sents[i]]
        else:
            cur.append(sents[i])
    chunks.append(" ".join(cur))
    return chunks

for i, c in enumerate(semantic_chunks(HANDBOOK)):
    print(f"CHUNK {i}:", c)

## INDEX lever 2 · Small-to-big (parent-document)

Search precise child chunks (sentences), but return the larger parent document for full context. You get the sharp match of small chunks and the rich context of large ones.

In [ ]:
def build_parent_child(corpus):
    parents, child_texts, child_parent = dict(corpus), [], []
    for pid, text in corpus.items():
        for sent in sentences(text):
            child_texts.append(sent)
            child_parent.append(pid)
    return parents, child_texts, child_parent

PARENTS, CHILD_TEXTS, CHILD_PARENT = build_parent_child(CORPUS)
CHILD_MATRIX = embed_many(CHILD_TEXTS)       # index over the small children

def small_to_big(query, k_children=4):
    sims = cosine_scores(CHILD_MATRIX, embed(query))
    order = np.argsort(-sims)[:k_children]
    seen, parents_out = set(), []
    for i in order:                          # map each child hit to its parent
        pid = CHILD_PARENT[i]
        if pid not in seen:
            seen.add(pid); parents_out.append(pid)
    return parents_out

hits = small_to_big("do gold members get a free extra bag?")
print("parents returned:", hits)
for h in hits:
    print(" -", h, "->", PARENTS[h][:70], "...")

## Base dense index (used by the query and rank levers below)

One embedding per document, so later techniques have something to search.

In [ ]:
MATRIX = embed_many(CHUNK_TEXTS)             # (num_docs, dim)
print("dense index:", MATRIX.shape)

def dense_search(query, matrix=None, ids=None, k=5):
    matrix = MATRIX if matrix is None else matrix
    ids = CHUNK_IDS if ids is None else ids
    sims = cosine_scores(matrix, embed(query))
    order = np.argsort(-sims)[:k]
    return [(ids[i], float(sims[i])) for i in order]

for cid, s in dense_search("how do I get my money back for a cancelled flight?", k=3):
    print(f"{s:.3f}  {cid}")

## INDEX lever 3 · Contextual Retrieval (Anthropic)

The disease: a chunk like "Gold members have the change fee waived" loses which airline and which policy once it is split out. The fix: an LLM writes a one-line context that situates each chunk, and we prepend it before embedding.

Below we compare raw vs contextual retrieval on a paraphrased query that shares almost no keywords with the source.

In [ ]:
def situate(doc_context, chunk_text):
    prompt = f"""<document>
{doc_context}
</document>

Here is a chunk from that document:
<chunk>
{chunk_text}
</chunk>

Give a short one-sentence context that situates this chunk within the document
(topic, key entities) so it can be understood on its own. Output only the sentence."""
    return chat(prompt, max_tokens=80)

CORPUS_TEXT = "\n".join(CORPUS.values())
CTX_TEXTS = [situate(CORPUS_TEXT, t) + " " + t for t in CHUNK_TEXTS]  # context + chunk
CTX_MATRIX = embed_many(CTX_TEXTS)

q = "Do premium frequent flyers avoid rebooking charges?"   # heavy paraphrase
print("RAW dense:")
for cid, s in dense_search(q, MATRIX, CHUNK_IDS, k=3):
    print(f"  {s:.3f}  {cid}")

print("CONTEXTUAL dense:")
sims = cosine_scores(CTX_MATRIX, embed(q))
for i in np.argsort(-sims)[:3]:
    print(f"  {float(sims[i]):.3f}  {CHUNK_IDS[i]}")

## RANK lever 1 · Hybrid search + Reciprocal Rank Fusion

Dense finds meaning, BM25 finds exact tokens (like the PNR `JX48Q2`). Fuse the two ranked lists with RRF, which rewards documents that rank high across lists.

$$\text{RRF}(d) = \sum_{r \in \text{lists}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

In [ ]:
from rank_bm25 import BM25Okapi

def tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

bm25 = BM25Okapi([tok(t) for t in CHUNK_TEXTS])

def bm25_search(query, k=5):
    scores = bm25.get_scores(tok(query))
    order = np.argsort(-scores)[:k]
    return [(CHUNK_IDS[i], float(scores[i])) for i in order]

def rrf(rank_lists, k=60):
    scores = {}
    for lst in rank_lists:
        for rank, cid in enumerate(lst):     # rank 0 is best
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(query, k=5):
    dense_ids  = [cid for cid, _ in dense_search(query, k=8)]
    sparse_ids = [cid for cid, _ in bm25_search(query, k=8)]
    return rrf([dense_ids, sparse_ids])[:k]

q = "change fee for PNR JX48Q2"               # exact code -> BM25 wins it
print("dense :", [cid for cid, _ in dense_search(q, k=3)])
print("bm25  :", [cid for cid, _ in bm25_search(q, k=3)])
print("hybrid:", hybrid_search(q, k=3))

## QUERY lever 1 · Multi-query / RAG-Fusion

One phrasing retrieves one slice of the truth. Generate several phrasings, retrieve each, and fuse with RRF.

In [ ]:
def gen_queries(q, n=3):
    out = chat(
        f"Generate {n} diverse alternative phrasings of this search query, "
        f"one per line, no numbering and no bullets.\n\nQuery: {q}",
        max_tokens=150, temperature=0.7,
    )
    variants = [ln.strip("-* ").strip() for ln in out.splitlines() if ln.strip()]
    return [q] + variants[:n]

def rag_fusion(query, k=4):
    variants = gen_queries(query, n=3)
    lists = [[cid for cid, _ in dense_search(v, k=6)] for v in variants]
    return variants, rrf(lists)[:k]

variants, fused = rag_fusion("waived rebooking charges for frequent flyers")
print("variants:")
for v in variants: print("  -", v)
print("fused:", fused)

## QUERY lever 2 · HyDE (Hypothetical Document Embeddings)

When a question and its answer share few words, embed a hypothetical answer instead of the question. An answer looks like the documents you want to find, so retrieval lands closer. The hypothetical can be factually wrong and still be right in shape.

In [ ]:
def hyde_passage(q):
    return chat(
        f"Write a short, plausible 2 to 3 sentence passage that would directly "
        f"answer this question. Do not hedge.\n\nQuestion: {q}",
        max_tokens=150, temperature=0.3,
    )

q = "What is the change fee policy for the highest tier members?"
hp = hyde_passage(q)
print("hypothetical answer:\n ", hp, "\n")
sims = cosine_scores(MATRIX, embed(hp))       # search with the hypothetical, not the question
for i in np.argsort(-sims)[:3]:
    print(f"  {float(sims[i]):.3f}  {CHUNK_IDS[i]}")

## QUERY lever 3 · Step-back prompting

Some questions are too specific to retrieve well. Ask the more general "step-back" question first to surface the governing policy, then combine both retrievals.

In [ ]:
def step_back_q(q):
    return chat(
        "Write the more general 'step-back' question about the underlying policy "
        f"behind this specific question. Output only the question.\n\nQuestion: {q}",
        max_tokens=60,
    )

q = "Can Rao change PNR JX48Q2 for free as a Gold member?"
sb = step_back_q(q)
print("step-back question:", sb)
combined = rrf([
    [cid for cid, _ in dense_search(q,  k=5)],
    [cid for cid, _ in dense_search(sb, k=5)],
])
print("combined retrieval:", combined[:4])

## RANK lever 2 · LLM reranking (listwise)

Retrieval scores are coarse. Give the model the shortlist and let it reorder by true relevance. In production a cross-encoder (Cohere Rerank, BGE-reranker) does this faster; an LLM reranker needs no extra service.

In [ ]:
def llm_rerank(query, candidates, k=5):
    numbered = "\n".join(f"[{i}] {t}" for i, (cid, t) in enumerate(candidates))
    out = chat(
        f"Query: {query}\n\nPassages:\n{numbered}\n\n"
        "List the passage numbers from most to least relevant to the query, "
        "comma-separated, numbers only.",
        max_tokens=80,
    )
    order = [int(x) for x in re.findall(r"\d+", out)]
    picked, seen = [], set()
    for i in order:
        if 0 <= i < len(candidates) and i not in seen:
            picked.append(candidates[i]); seen.add(i)
    for i in range(len(candidates)):           # append anything the model dropped
        if i not in seen:
            picked.append(candidates[i])
    return picked[:k]

cands = [(cid, CORPUS[cid]) for cid in CHUNK_IDS]
for cid, _ in llm_rerank("free seat selection for silver members", cands, k=3):
    print(" -", cid)

## RANK lever 3 · MMR (relevance vs diversity)

Top-k by pure similarity often returns near-duplicates. Maximal Marginal Relevance trades a little relevance for coverage.

$$\text{MMR} = \arg\max_{d \notin S}\left[\lambda\,\text{sim}(d,q) - (1-\lambda)\max_{d' \in S}\text{sim}(d,d')\right]$$

In [ ]:
def mmr(query, cand_ids, k=4, lam=0.7):
    qv = embed(query)
    cv = embed_many([CORPUS[c] for c in cand_ids])
    sim_q = cosine_scores(cv, qv)
    selected, remaining = [], list(range(len(cand_ids)))
    while remaining and len(selected) < k:
        best, best_s = None, -1e9
        for i in remaining:
            redundancy = max((cosine(cv[i], cv[j]) for j in selected), default=0.0)
            score = lam * sim_q[i] - (1 - lam) * redundancy
            if score > best_s:
                best_s, best = score, i
        selected.append(best); remaining.remove(best)
    return [cand_ids[i] for i in selected]

print("MMR selection:", mmr("baggage and check in rules", CHUNK_IDS, k=3))

## RANK lever 4 · Contextual compression

Retrieved chunks carry filler. Strip each down to the sentences that actually bear on the query before they reach the model. Less noise, fewer tokens, and it counters the "lost in the middle" failure.

In [ ]:
def compress(query, chunks):
    joined = "\n---\n".join(chunks)
    return chat(
        f"Query: {query}\n\nContext:\n{joined}\n\n"
        "Copy only the sentences from the context that help answer the query. "
        "If none apply, output NONE.",
        max_tokens=250,
    )

ids = hybrid_search("how long do refunds take", k=3)
ctx = [CORPUS[c] for c in ids]
print("before compression:", ids)
print("after compression:\n", compress("how long do refunds take", ctx))

## Compose: a strong retriever

Stack the levers that earned their place: multi-query expansion, hybrid retrieval per query, RRF fusion, then an LLM rerank. Then answer, grounded and cited.

This is the Compounding Recipe from the deck, in code: base (hybrid) + query transform (multi-query) + rank (rerank).

In [ ]:
def strong_retrieve(query, k=4):
    variants = gen_queries(query, n=2)                  # QUERY lever
    lists = []
    for v in variants:                                  # hybrid per variant
        lists.append([cid for cid, _ in dense_search(v, k=6)])
        lists.append([cid for cid, _ in bm25_search(v, k=6)])
    fused = rrf(lists)[:8]                               # RANK: fuse
    reranked = llm_rerank(query, [(c, CORPUS[c]) for c in fused], k=k)  # RANK: rerank
    return [c for c, _ in reranked]

def answer(query):
    ids = strong_retrieve(query, k=4)
    ctx = "\n\n".join(f"[{c}] {CORPUS[c]}" for c in ids)
    prompt = (
        "Answer only from the context. Cite the [id] tags you used. "
        "If the answer is not present, say you do not know.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {query}"
    )
    return ids, chat(prompt, max_tokens=300)

ids, ans = answer("As a Gold member, will I pay to change my cancelled flight, and how long for a refund?")
print("retrieved:", ids, "\n")
print(ans)

## What changes in production

| In this notebook | In production |
|---|---|
| plain numpy index in memory | a real vector store (OpenSearch, pgvector, or Bedrock Knowledge Bases) |
| access keys via `aws configure` | an IAM role, least privilege, no hardcoded secrets |
| LLM reranker | a dedicated cross-encoder reranker (Cohere, BGE) for speed |
| one call at a time | batched and async embedding and retrieval, with retries and backoff |
| no measurement | recall@k, MRR, nDCG on a labeled set, wired into CI |
| contexts recomputed each run | contextual chunks built once at index time, with prompt caching |

Recap of the levers you now have as code:

| Lever | Functions |
|---|---|
| INDEX | `semantic_chunks`, `small_to_big`, `situate` (contextual) |
| QUERY | `gen_queries` / `rag_fusion`, `hyde_passage`, `step_back_q` |
| RANK | `hybrid_search` + `rrf`, `llm_rerank`, `mmr`, `compress` |
| Compose | `strong_retrieve`, `answer` |